# Mã 21: phát hiện OOD và hiệu chỉnh độ tin cậy trên BCN20000 (Colab + Google Drive)

**Trước khi chạy:**
1. Mở notebook trên Google Colab.
2. **Runtime → Change runtime type → GPU T4.**
3. Chạy cell 0. Colab sẽ hỏi quyền truy cập Google Drive: bạn tự đăng nhập và cho phép.
4. **Runtime → Run all.**

**Lưu trên Google Drive** (thư mục `MyDrive/ma21_bcn20000`):
- `data/`: tệp zip train (6,37 GB), CSV, và file ghi nhận đã kiểm MD5 2293c9c654a3f76d6506a57ad17cddd1.
- `checkpoints/seedN/`: `last.pt` (lưu sau mỗi epoch) và `best_model.pt`.
- `outputs/`: results_seed*.json, predictions_seed*.csv và summary_per_seed.csv.

Cần khoảng 6,5 GB dung lượng Drive. Kiểm tra trong Drive, mục Storage.

**Trên ổ của máy ảo** (tạm, tạo lại mỗi phiên): ảnh đã giải nén và cache ảnh 256 px. Mỗi phiên mới mất vài phút để giải nén và tạo cache.

**Chạy tiếp khi bị ngắt:** chạy lại notebook. Seed đã xong được bỏ qua, seed đang dở sẽ tiếp tục từ epoch đã lưu gần nhất.

**Lưu ý:** Colab miễn phí thường chỉ có một GPU. Phiên có thể tối đa khoảng 12 giờ. HOLDOUT mặc định là AK (chưa chốt, xem STATUS.md). Tệp test không dùng vì không có nhãn.

In [ ]:
# Cell 0: gắn Google Drive (Colab) và ghi đè cấu hình tùy chọn
import os
if os.path.isdir('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
# os.environ["SEEDS"] = "0,1,2"
# os.environ["EPOCHS"] = "12"
# os.environ["HOLDOUT"] = "AK"
# os.environ["BATCH"] = "32"   # batch mỗi GPU
# os.environ["CACHE"] = "1"    # 1: thu nhỏ ảnh một lần (khuyến nghị)
# os.environ["RESUME"] = "1"   # 1: tiếp tục từ checkpoint và bỏ qua seed đã xong

In [ ]:
# %% 1. Cấu hình và tải dữ liệu
import os
import sys
import json
import time
import hashlib
import zipfile
import subprocess
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd


def env(name, default, kind=str):
    v = os.environ.get(name)
    if v is None:
        return default
    if kind is bool:
        return v.strip().lower() in ('1', 'true', 'yes')
    if kind is list:
        return [x.strip() for x in v.split(',') if x.strip()]
    return kind(v)


IN_COLAB = os.path.isdir('/content') and not os.path.isdir('/kaggle')
WORK = env('WORK', '/content' if IN_COLAB else '/kaggle/working')
DRIVE_ROOT = env('DRIVE_ROOT', '/content/drive/MyDrive/ma21_bcn20000' if IN_COLAB else f'{WORK}/ma21_bcn20000')
LOCAL = env('LOCAL', '/content/local' if IN_COLAB else f'{WORK}/local')
DATA_DIR = f'{DRIVE_ROOT}/data'
ZIP_URL = 'https://ndownloader.figshare.com/files/45833178'   # BCN_20k_train.zip, 6368901290 byte
ZIP_MD5 = '2293c9c654a3f76d6506a57ad17cddd1'                  # checksum chính thức trên figshare
CSV_URL = 'https://ndownloader.figshare.com/files/49291684'   # bcn_20k_train.csv
HOLDOUT = env('HOLDOUT', ['AK'], list)
SEEDS = [int(s) for s in env('SEEDS', ['0'], list)]
EPOCHS = env('EPOCHS', 12, int)
BATCH = env('BATCH', 32, int)              # batch MỖI GPU; batch tổng = BATCH x số GPU
LR = env('LR', 3e-4, float)
IMG = env('IMG', 224, int)
RESIZE = env('RESIZE', 256, int)
MC_PASSES = env('MC_PASSES', 20, int)
BOOT = env('BOOT', 500, int)
PATIENCE = env('PATIENCE', 3, int)
PRETRAINED = env('PRETRAINED', True, bool)
NUM_WORKERS = env('NUM_WORKERS', min(8, os.cpu_count() or 4), int)
RESUME = env('RESUME', True, bool)                                     # tiếp tục từ checkpoint và bỏ qua seed đã xong
STOP_AFTER_EPOCH = env('STOP_AFTER_EPOCH', -1, int)                    # CHỈ để thử nghiệm: dừng sau epoch này
DATA_CSV = env('DATA_CSV', f'{DATA_DIR}/bcn_20k_train.csv')
IMAGE_DIR = env('IMAGE_DIR', f'{LOCAL}/train')
CACHE = env('CACHE', True, bool)                                       # thu nhỏ ảnh 1 lần, tránh giải mã JPEG 1024 px mỗi epoch
CACHE_DIR = env('CACHE_DIR', f'{LOCAL}/cache{RESIZE}')
CKPT_DIR = env('CKPT_DIR', f'{DRIVE_ROOT}/checkpoints')
OUT = env('OUT', f'{DRIVE_ROOT}/outputs')


def md5_of(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def ensure_data():
    if 'DATA_CSV' in os.environ and 'IMAGE_DIR' in os.environ:
        print('Dùng dữ liệu cục bộ:', DATA_CSV, IMAGE_DIR)
        return
    if IN_COLAB and not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('Chưa gắn Google Drive. Chạy cell gắn Drive trước.')
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    if not Path(DATA_CSV).exists():
        subprocess.run(['wget', '-q', '-O', DATA_CSV, CSV_URL], check=True)

    marker = Path(LOCAL) / 'extracted.ok'
    if marker.exists() and Path(IMAGE_DIR).is_dir():
        print('Đã giải nén trên máy ảo:', IMAGE_DIR)
        return

    zip_path = Path(DATA_DIR) / 'BCN_20k_train.zip'
    verified = Path(DATA_DIR) / 'zip_md5_ok.txt'
    already_ok = zip_path.exists() and verified.exists() and verified.read_text().strip() == ZIP_MD5
    if not already_ok:
        if not (zip_path.exists() and md5_of(zip_path) == ZIP_MD5):
            subprocess.run(['wget', '-c', '--progress=dot:giga', '-O', str(zip_path), ZIP_URL], check=True)
        got = md5_of(zip_path)
        if got != ZIP_MD5:
            raise RuntimeError(f'MD5 không khớp: {got} != {ZIP_MD5}. Xóa tệp zip trên Drive và chạy lại.')
        verified.write_text(ZIP_MD5)
    print('MD5 khớp:', ZIP_MD5)

    Path(LOCAL).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(IMAGE_DIR)
    marker.write_text('ok')
    print('Giải nén xong:', IMAGE_DIR)

In [ ]:
# %% 2. Chia dữ liệu theo lesion_id và class-holdout
from sklearn.model_selection import StratifiedGroupKFold


def index_images(root):
    idx = {}
    for dp, _, fns in os.walk(root):
        for f in fns:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                idx.setdefault(f, os.path.join(dp, f))
    return idx


def build_split(seed):
    df = pd.read_csv(DATA_CSV)
    if 'split' in df.columns:
        df = df[df['split'].astype(str).str.lower() == 'train']
    idx = index_images(IMAGE_DIR)
    miss = df.loc[~df['bcn_filename'].isin(list(idx.keys())), 'bcn_filename']
    if len(miss):
        raise FileNotFoundError(f'Thiếu {len(miss)} ảnh, ví dụ: {list(miss[:3])}')
    df = df.assign(path=df['bcn_filename'].map(idx)).reset_index(drop=True)

    ood = df[df['diagnosis'].isin(HOLDOUT)].reset_index(drop=True)
    idd = df[~df['diagnosis'].isin(HOLDOUT)].reset_index(drop=True)
    classes = sorted(idd['diagnosis'].unique().tolist())
    cmap = {c: i for i, c in enumerate(classes)}
    y = idd['diagnosis'].map(cmap).to_numpy()
    g = idd['lesion_id'].to_numpy()

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    rest, te = next(outer.split(np.zeros(len(y)), y, g))          # ~80% còn lại, ~20% id_test
    inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=seed)
    a, b = next(inner.split(np.zeros(len(rest)), y[rest], g[rest]))  # ~70% train, ~10% val
    tr, va = rest[a], rest[b]

    gs_tr, gs_va, gs_te = set(g[tr]), set(g[va]), set(g[te])
    assert not (gs_tr & gs_va) and not (gs_tr & gs_te) and not (gs_va & gs_te), 'Rò rỉ lesion giữa các tập'

    return dict(idd=idd, ood=ood, y=y, g=g, classes=classes, tr=tr, va=va, te=te)

In [ ]:
# %% 3. Dataset, mô hình và huấn luyện (có checkpoint và tiếp tục)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = T.Compose([T.Resize(RESIZE), T.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.Resize(RESIZE), T.CenterCrop(IMG), T.ToTensor(), T.Normalize(MEAN, STD)])


class ImgDS(Dataset):
    def __init__(self, paths, labels, tf):
        self.p = list(paths)
        self.y = np.asarray(labels, dtype=np.int64)
        self.tf = tf

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        im = Image.open(self.p[i])
        im.draft('RGB', (RESIZE * 2, RESIZE * 2))   # giải mã JPEG ở độ phân giải thấp, nhanh hơn
        im = im.convert('RGB')
        return self.tf(im), self.y[i]


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def get_rng():
    st = {'torch': torch.get_rng_state(), 'np': np.random.get_state(), 'py': random.getstate()}
    if torch.cuda.is_available():
        st['cuda'] = torch.cuda.get_rng_state_all()
    return st


def restore_rng(st):
    torch.set_rng_state(st['torch'])
    np.random.set_state(st['np'])
    random.setstate(st['py'])
    if 'cuda' in st and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st['cuda'])


def save_atomic(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)          # ghi xong mới thay tệp cũ, tránh tệp dở khi mất kết nối Drive


def build_model(n_cls):
    if PRETRAINED:
        m = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    else:
        m = torchvision.models.efficientnet_b0(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, n_cls))
    return m


@torch.no_grad()
def get_feats_logits(model, loader, device, use_amp):
    model.eval()
    feats, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            f = torch.flatten(model.avgpool(model.features(x)), 1)
        feats.append(f.float().cpu())
        ys.append(torch.as_tensor(y))
    feats = torch.cat(feats)
    ys = torch.cat(ys)
    logits = model.classifier(feats.to(device)).float().cpu()
    return feats, logits, ys


def train_model(seed, ds_tr, ds_va, n_cls, class_w, device, use_amp):
    set_seed(seed)
    model = build_model(n_cls).to(device)
    ck_dir = Path(CKPT_DIR) / f'seed{seed}'
    best_path = ck_dir / 'best_model.pt'
    last_path = ck_dir / 'last.pt'
    if RESUME and best_path.exists():
        ck = torch.load(best_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['state'])
        print(f'  đã có {best_path.name}, bỏ qua huấn luyện')
        return model, ck['hist'], ck['best_ep']

    # Dùng tất cả GPU có sẵn: DataParallel chia batch ra từng card (chỉ dùng cho forward khi huấn luyện).
    # Batch tổng = BATCH x số GPU, để mỗi card vẫn nhận BATCH mẫu.
    ngpu = torch.cuda.device_count() if device.type == 'cuda' else 1
    net = nn.DataParallel(model) if ngpu > 1 else model
    gbs = BATCH * ngpu
    tr_dl = DataLoader(ds_tr, batch_size=gbs, shuffle=True, num_workers=NUM_WORKERS,
                       pin_memory=device.type == 'cuda', drop_last=False)
    va_dl = DataLoader(ds_va, batch_size=gbs * 2, shuffle=False, num_workers=NUM_WORKERS,
                       pin_memory=device.type == 'cuda')
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(tr_dl))
    crit = nn.CrossEntropyLoss(weight=class_w.to(device))
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    best = (float('inf'), None, -1)
    bad, hist, start = 0, [], 0
    if RESUME and last_path.exists():
        ck = torch.load(last_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched'])
        if use_amp and ck['scaler'] is not None:
            scaler.load_state_dict(ck['scaler'])
        best = (ck['best_loss'], ck['best_state'], ck['best_ep'])
        bad, hist, start = ck['bad'], ck['hist'], ck['epoch'] + 1
        restore_rng(ck['rng'])
        if bad >= PATIENCE:
            start = EPOCHS            # đã dừng sớm trước khi ngắt
        print(f'  tiếp tục từ epoch {start} (đọc {last_path.name})')

    for ep in range(start, EPOCHS):
        t0 = time.time()
        model.train()
        tot, n = 0.0, 0
        for x, y in tr_dl:
            x, y = x.to(device, non_blocking=True), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = crit(net(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y)
            n += len(y)
        _, vlog, vy = get_feats_logits(model, va_dl, device, use_amp)
        vnll = F.cross_entropy(vlog, vy).item()
        hist.append(dict(epoch=ep, train_loss=tot / n, val_nll=vnll, sec=round(time.time() - t0, 1)))
        print(f'  epoch {ep:02d} train_loss {tot / n:.4f} val_nll {vnll:.4f} ({hist[-1]["sec"]}s)')
        if vnll < best[0] - 1e-4:
            best = (vnll, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep)
            bad = 0
        else:
            bad += 1

        save_atomic({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                     'sched': sched.state_dict(), 'scaler': scaler.state_dict() if use_amp else None,
                     'best_loss': best[0], 'best_state': best[1], 'best_ep': best[2],
                     'bad': bad, 'hist': hist, 'rng': get_rng(), 'seed': seed, 'epochs': EPOCHS}, last_path)
        if STOP_AFTER_EPOCH == ep:
            raise SystemExit(f'[thử nghiệm] dừng sau epoch {ep}, checkpoint đã lưu')
        if bad >= PATIENCE:
            print('  dừng sớm')
            break

    model.load_state_dict(best[1])
    save_atomic({'state': best[1], 'hist': hist, 'best_ep': best[2]}, best_path)
    return model, hist, best[2]


def fit_temperature(logits, ys):
    logt = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logt], lr=0.1, max_iter=200, line_search_fn='strong_wolfe')

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / logt.exp(), ys)
        loss.backward()
        return loss

    opt.step(closure)
    return float(logt.exp().item())


@torch.no_grad()
def mc_probs(model, feats, T_, passes, device):
    head = model.classifier
    head.eval()
    head[0].train()                      # chỉ bật dropout, giữ BatchNorm ở chế độ eval
    f = feats.to(device)
    ps = [torch.softmax(head(f) / T_, dim=1).cpu() for _ in range(passes)]
    head.eval()
    return torch.stack(ps).mean(0)

In [ ]:
# %% 4. Chỉ số
def ece_score(P, ys, n_bins=15):
    conf, pred = P.max(1)
    acc = (pred == ys).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return float(e)


def nll(P, ys):
    return float(-torch.log(P[torch.arange(len(ys)), ys].clamp_min(1e-12)).mean().item())


def brier(P, ys):
    oh = F.one_hot(ys, P.shape[1]).float()
    return float(((P - oh) ** 2).sum(1).mean().item())


def clf_metrics(P, ys):
    pred = P.argmax(1).numpy()
    yy = ys.numpy()
    return dict(acc=float((pred == yy).mean()), bal_acc=float(balanced_accuracy_score(yy, pred)),
                nll=nll(P, ys), brier=brier(P, ys), ece=ece_score(P, ys))


def auroc_fpr(sid, sood):
    yy = np.r_[np.ones(len(sid)), np.zeros(len(sood))]
    s = np.r_[sid, sood]
    auc = float(roc_auc_score(yy, s))
    thr = np.percentile(sid, 5)                     # giữ 95% ID
    fpr = float((sood >= thr).mean())
    return auc, fpr


def lesion_bootstrap(metric_fn, gid, good, B, seed):
    rng = np.random.default_rng(seed)
    ug_i, ug_o = np.unique(gid), np.unique(good)
    mi = {k: np.flatnonzero(gid == k) for k in ug_i}
    mo = {k: np.flatnonzero(good == k) for k in ug_o}
    vals = []
    for _ in range(B):
        ii = np.concatenate([mi[k] for k in rng.choice(ug_i, len(ug_i))])
        oo = np.concatenate([mo[k] for k in rng.choice(ug_o, len(ug_o))])
        vals.append(metric_fn(ii, oo))
    out = {}
    for k in vals[0]:
        arr = np.array([v[k] for v in vals])
        out[k] = [float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))]
    return out


def env_info():
    info = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
                cuda=torch.cuda.is_available(), cpu_count=os.cpu_count(), in_colab=IN_COLAB)
    if torch.cuda.is_available():
        info['gpu'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        info['n_gpu'] = torch.cuda.device_count()
    return info

In [ ]:
# %% 5. Chạy một seed
def run_seed(seed, device, use_amp):
    res_path = Path(OUT) / f'results_seed{seed}.json'
    if RESUME and res_path.exists():
        print(f'seed {seed}: đã có {res_path.name}, bỏ qua')
        return json.load(open(res_path, encoding='utf-8'))

    S = build_split(seed)
    idd, ood, y = S['idd'], S['ood'], S['y']
    K = len(S['classes'])
    tr, va, te = S['tr'], S['va'], S['te']
    print(f'seed {seed}: train {len(tr)}, val {len(va)}, id_test {len(te)}, ood {len(ood)}, lớp ID {S["classes"]}')

    counts = np.bincount(y[tr], minlength=K)
    class_w = torch.tensor(len(tr) / (K * np.maximum(counts, 1)), dtype=torch.float32)

    ds_tr = ImgDS(idd['path'].values[tr], y[tr], tf_train)
    ds_va = ImgDS(idd['path'].values[va], y[va], tf_eval)
    model, hist, best_ep = train_model(seed, ds_tr, ds_va, K, class_w, device, use_amp)
    print(f'  best epoch {best_ep}')

    dl = lambda ds: DataLoader(ds, batch_size=BATCH * 2, shuffle=False, num_workers=NUM_WORKERS,
                               pin_memory=device.type == 'cuda')
    fva, Lva, yva = get_feats_logits(model, dl(ds_va), device, use_amp)
    T_ = fit_temperature(Lva, yva)
    print(f'  temperature T = {T_:.4f}')

    ds_te = ImgDS(idd['path'].values[te], y[te], tf_eval)
    ds_ood = ImgDS(ood['path'].values, np.full(len(ood), -1), tf_eval)
    fte, Lte, yte = get_feats_logits(model, dl(ds_te), device, use_amp)
    food, Lood, _ = get_feats_logits(model, dl(ds_ood), device, use_amp)

    pr_te = torch.softmax(Lte, 1)
    pt_te = torch.softmax(Lte / T_, 1)
    mc_te = mc_probs(model, fte, T_, MC_PASSES, device)
    mc_ood = mc_probs(model, food, T_, MC_PASSES, device)

    fns = {
        'msp_raw': lambda L: torch.softmax(L, 1).max(1).values,
        'msp_ts': lambda L: torch.softmax(L / T_, 1).max(1).values,
        'energy_raw': lambda L: torch.logsumexp(L, 1),
        'energy_ts': lambda L: T_ * torch.logsumexp(L / T_, 1),
    }
    sc_id = {k: f(Lte).numpy() for k, f in fns.items()}
    sc_ood = {k: f(Lood).numpy() for k, f in fns.items()}
    sc_id['mc_msp'] = mc_te.max(1).values.numpy()
    sc_ood['mc_msp'] = mc_ood.max(1).values.numpy()

    ood_res = {k: dict(zip(['auroc', 'fpr95'], auroc_fpr(sc_id[k], sc_ood[k]))) for k in sc_id}

    gid = S['g'][te]
    good = ood['lesion_id'].values

    ci_cal = lesion_bootstrap(
        lambda ii, oo: dict(ece_ts=ece_score(pt_te[torch.as_tensor(ii)], yte[torch.as_tensor(ii)]),
                            nll_ts=nll(pt_te[torch.as_tensor(ii)], yte[torch.as_tensor(ii)])),
        gid, good, BOOT, seed)
    ci_ood = lesion_bootstrap(
        lambda ii, oo: {f'{m}_{k}': v for k in sc_id for m, v in zip(
            ['auroc', 'fpr95'], auroc_fpr(sc_id[k][ii], sc_ood[k][oo]))},
        gid, good, BOOT, seed + 1)

    res = dict(
        seed=seed, classes=S['classes'], holdout=HOLDOUT, n_train=int(len(tr)), n_val=int(len(va)),
        n_id_test=int(len(te)), n_ood=int(len(ood)), n_lesions_train=int(len(set(S['g'][tr]))),
        best_epoch=best_ep, temperature=T_, history=hist,
        id_metrics_raw=clf_metrics(pr_te, yte), id_metrics_ts=clf_metrics(pt_te, yte),
        ood=ood_res, ci_calibration_ts=ci_cal, ci_ood=ci_ood,
        env=env_info(), config=dict(EPOCHS=EPOCHS, BATCH=BATCH, LR=LR, IMG=IMG, RESIZE=RESIZE,
                                    MC_PASSES=MC_PASSES, BOOT=BOOT, PATIENCE=PATIENCE,
                                    PRETRAINED=PRETRAINED, HOLDOUT=HOLDOUT),
    )
    os.makedirs(OUT, exist_ok=True)
    with open(res_path, 'w', encoding='utf-8') as f:
        json.dump(res, f, ensure_ascii=False, indent=2)

    pred_te = pt_te.argmax(1).numpy()
    df_id = pd.DataFrame({
        'bcn_filename': idd['bcn_filename'].values[te], 'lesion_id': gid,
        'diagnosis': idd['diagnosis'].values[te], 'is_ood': 0, 'y': yte.numpy(),
        'pred_raw': pr_te.argmax(1).numpy(), 'pred_ts': pred_te,
        'pmax_raw': pr_te.max(1).values.numpy(), 'pmax_ts': pt_te.max(1).values.numpy(),
        'mc_msp': sc_id['mc_msp'], 'energy_ts': sc_id['energy_ts']})
    df_ood = pd.DataFrame({
        'bcn_filename': ood['bcn_filename'].values, 'lesion_id': good,
        'diagnosis': ood['diagnosis'].values, 'is_ood': 1, 'y': -1,
        'pred_raw': torch.softmax(Lood, 1).argmax(1).numpy(),
        'pred_ts': torch.softmax(Lood / T_, 1).argmax(1).numpy(),
        'pmax_raw': torch.softmax(Lood, 1).max(1).values.numpy(),
        'pmax_ts': torch.softmax(Lood / T_, 1).max(1).values.numpy(),
        'mc_msp': sc_ood['mc_msp'], 'energy_ts': sc_ood['energy_ts']})
    pd.concat([df_id, df_ood]).to_csv(os.path.join(OUT, f'predictions_seed{seed}.csv'), index=False)

    print('  ID test (sau TS):', {k: round(v, 4) for k, v in res['id_metrics_ts'].items()})
    print('  OOD (AUROC, FPR95):', {k: (round(v['auroc'], 4), round(v['fpr95'], 4)) for k, v in ood_res.items()})
    return res

In [ ]:
# %% 6. Tổng hợp và chạy
def summarize():
    import glob
    rows = []
    for p in sorted(glob.glob(os.path.join(OUT, 'results_seed*.json'))):
        r = json.load(open(p, encoding='utf-8'))
        row = {'seed': r['seed'], 'T': r['temperature'], 'best_epoch': r['best_epoch']}
        for k, v in r['id_metrics_raw'].items():
            row[f'raw_{k}'] = v
        for k, v in r['id_metrics_ts'].items():
            row[f'ts_{k}'] = v
        for k, v in r['ood'].items():
            row[f'auroc_{k}'] = v['auroc']
            row[f'fpr95_{k}'] = v['fpr95']
        rows.append(row)
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(OUT, 'summary_per_seed.csv'), index=False)
        print(df.T.to_string())
        print('\nTrung bình ± độ lệch chuẩn qua các seed:')
        print(df.drop(columns=['seed']).agg(['mean', 'std']).T.to_string())


def _resize_one(job):
    src, dst = job
    im = Image.open(src)
    im.draft('RGB', (RESIZE * 2, RESIZE * 2))
    im = T.Resize(RESIZE)(im.convert('RGB'))
    im.save(dst, quality=95)
    return dst


def make_cache(src_root, dst_dir):
    """Thu nhỏ mọi ảnh train về RESIZE px một lần, lưu JPEG chất lượng cao.
    Mỗi epoch sau đó chỉ giải mã ảnh 256 px (nhanh hơn nhiều so với ảnh 1024 px)."""
    from multiprocessing import Pool
    dst = Path(dst_dir)
    dst.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(DATA_CSV)
    if 'split' in df.columns:
        df = df[df['split'].astype(str).str.lower() == 'train']
    idx = index_images(src_root)
    todo = []
    for f in df['bcn_filename']:
        if not (dst / f).exists():
            todo.append((idx[f], str(dst / f)))
    if not todo:
        print('Cache đã đủ:', dst_dir)
        return str(dst)
    print(f'Tạo cache {len(todo)} ảnh -> {dst_dir}')
    with Pool(max(1, NUM_WORKERS)) as p:
        for i, _ in enumerate(p.imap_unordered(_resize_one, todo, chunksize=32)):
            if (i + 1) % 2000 == 0:
                print('  đã xong', i + 1)
    print('Cache xong:', dst_dir)
    return str(dst)


def main():
    global IMAGE_DIR
    os.makedirs(OUT, exist_ok=True)
    os.makedirs(CKPT_DIR, exist_ok=True)
    ensure_data()
    if CACHE:
        IMAGE_DIR = make_cache(IMAGE_DIR, CACHE_DIR)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = device.type == 'cuda'
    print('device:', device, '| torch:', torch.__version__, '| holdout:', HOLDOUT, '| seeds:', SEEDS)
    print('drive:', DRIVE_ROOT, '| checkpoint:', CKPT_DIR, '| kết quả:', OUT)
    for s in SEEDS:
        run_seed(s, device, use_amp)
    summarize()


if __name__ == '__main__':
    main()